# NB_Silver_Sellers — Bronze to Silver Transformation (Upgraded v2.1 — Option 2 POC / Learning Mode)

## Section 1 — Purpose, Business Context & Engineering Decisions
- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.sellers` → ADLS Bronze Parquet exposed via Fabric Shortcut (`Files/bronze/catalog/sellers/`).
- **Grain:** One row per marketplace seller (`seller_id`, `3,095` rows).
- **Target Silver Table:** `silver.sellers` (managed Delta table registered in the Fabric Lakehouse Catalog).
- **Quarantine Table:** `silver._quarantine_sellers` (for **Row-Level Entity Failures** ONLY — e.g. invalid `seller_id`, invalid 5-digit ZIP, or invalid Brazilian UF state code).
- **Execution Environment Note:** Designed to run on **Microsoft Fabric (`synapse_pyspark` kernel)** where `spark` is pre-configured automatically. Do **not** add `%pip install pyspark` or run this locally without a Fabric/SparkSession.

### Key Data Quality Findings Driving This Notebook (`CONFIRMED` on `3,095` source rows)
1. **Entity Keys Are Clean (`0` Row-Level Quarantine expected):** All `3,095` `seller_id` values are unique 32-char lowercase hex strings; all `3,095` `seller_zip_code_prefix` values are 5 digits; all `23` distinct `seller_state` codes belong to Brazil's 27 official UF codes.
2. **Option 2 Applied — Auditable Static `(ZIP, State) -> City` Override (POC / Learning Mode):**
   - Those 2 sellers with corrupted `seller_city` values (`vendas@creditparts.com.br` and `04482255`) own **39 real sales lines in `order_items`** worth **`7,532.41` BRL** (`38` lines / `7,115.96` BRL for `vendas@creditparts.com.br`; `1` line / `416.45` BRL for `04482255`).
   - Dropping/quarantining those seller rows out of `silver.sellers` would immediately **break Referential Integrity with `silver.order_items`**.
   - **Approved Option 2 Solution:** Keep all `3,095` valid sellers in `silver.sellers`, preserve the original raw input in `seller_city_raw`, and replace the 2 verified corrupted cities statically via `VERIFIED_ZIP_CITY_OVERRIDES`:
     - `("87025", "PR") -> "maringa"` (verified across `82` rows in `dbo.dim_geolocation`: `77` `'maringa'` + `5` `'maringá'`, 100% `PR`).
     - `("22790", "RJ") -> "rio de janeiro"` (verified across `687` rows in `dbo.dim_geolocation`: 100% `'rio de janeiro'`, `RJ`).
   - Flag them transparently as `seller_city_dq_status = 'CORRECTED_FROM_ZIP_LOOKUP'` and `seller_city_source = 'STATIC_ZIP_OVERRIDE_POC'`.
   - **Future-Safe Guardrail:** Any new/unknown corrupted city arriving in future loads that is NOT in `VERIFIED_ZIP_CITY_OVERRIDES` safely falls back to `NULL` with `seller_city_dq_status = 'INVALID_CITY_NULLIFIED'` and `seller_city_source = 'NULLIFIED_INVALID_SOURCE'`. In a later production **Stage-2 / Gold `dim_seller`** pipeline, this static dictionary can be upgraded to a dynamic 1:1 `LEFT JOIN` against `silver.geolocation_zip_reference`.
3. **Expanded City Review Detection (`36` rows flagged `REVIEW_REQUIRED`):** Catches punctuation/composite strings (`lages - sc`, `auriflama/sp`, `arraial d'ajuda (porto seguro)`), non-ASCII/accents (`são paulo`, `santa barbara d´oeste`), bare/trailing state codes (`sp` ×4, `brasilia df`, `sao paulo sp`, `angra dos reis rj`, `aguas claras df`), and abbreviations (`sbc`) — without false-flagging valid 3-letter cities (`itu`, `uba`, `poa`, `ipe`, `jau`).
4. **Known Cross-Table Note (Handled in Phase 5 / Gold, NOT Joined Here):** `35` sellers (`646` order items) have `seller_state` conflicting with `dim_geolocation`, and `7` sellers have ZIPs absent from `dim_geolocation`. Keep Stage-1 Silver single-table and resolve via `LEFT JOIN` + DQ check in Phase 5 / Gold.

> **WARNING:** **DO NOT EXECUTE THE WRITE STEP UNTIL THE TRANSFORMATION HAS BEEN REVIEWED AND APPROVED.** (`WRITE_ENABLED = False` by default.)

## Section 2 — Configuration & Parameters
### ماذا تفعل هذه الخلية؟ (For Junior Data Engineers)
- نستخدم **مسارات نسبية (Relative Lakehouse Paths)** مثل `Files/bronze/catalog/sellers/...` لأن هذا الدفتر مرتبط بـ Default Lakehouse (`Retail_Analytics_Lakehouse`)، وبالتالي لا نحتاج لكتابة Workspace/Lakehouse GUIDs الثابتة.
- **حماية من خلط الـ Snapshots:** نحدد `SOURCE_FILE_NAME` كـ Parameter (يمرره Fabric Data Factory Pipeline مستقبلاً) بدلاً من قراءة المجلد كاملاً بشكل أعمى حتى لا نجمع لقطتين كاملتين (Full Snapshots) فوق بعضهما بالخطأ!
- **`SILVER_SCHEMA = "silver"`:** نحدّد اسم الـ Schema صراحةً ليُنشأ تلقائياً وقت الكتابة (`CREATE SCHEMA IF NOT EXISTS silver`) فلا يظهر خطأ `SCHEMA_NOT_FOUND`.
- **`VERIFIED_ZIP_CITY_OVERRIDES` (تطبيق الخيار الثاني — Static POC):** قاموس استثناءات موثّق لتعويض القيمتين الفاسدتين بناءً على فحص `dbo.dim_geolocation` بدلاً من عمل `JOIN` مع جدول Geolocation الآن، مع الإبقاء على `seller_city_raw` وتسجيل المصدر.
- **`WRITE_ENABLED = False`:** قفل أمان يمنع أي كتابة في Silver أو Quarantine أثناء التشغيل التجريبي (`Dry Run`).

In [ ]:
from pyspark.sql import functions as F

# =============================================================================
# 1. SAFETY LOCK — Run once as Dry Run (False), then set True to persist Delta
# =============================================================================
WRITE_ENABLED = False

# =============================================================================
# 2. PIPELINE PARAMETERS (Override via Fabric Pipeline parameters in production)
# =============================================================================
BRONZE_DIR = "Files/bronze/catalog/sellers"
# Specify the exact Full Snapshot file to avoid mixing multiple historical snapshots
SOURCE_FILE_NAME = "sellers_20261121_013305_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"
BRONZE_PATH = f"{BRONZE_DIR}/{SOURCE_FILE_NAME}"

INGESTION_RUN_ID = "b0e93f82-84a0-4668-9a8c-9ae1da7dd183"
SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.sellers"

# Lakehouse Catalog schema & table names (Schemas-enabled Lakehouse)
SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.sellers"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_sellers"

# Official 27 Brazilian Federal Units (UF) whitelist — stricter than ^[A-Z]{2}$
VALID_BR_STATES = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

# Verified Static (ZIP, State) -> City overrides for corrupted seller_city values (Option 2 POC / Learning)
# Verified against dbo.dim_geolocation:
#   ("87025", "PR") -> 82 rows (77 'maringa' + 5 'maringá', 100% PR)
#   ("22790", "RJ") -> 687 rows (100% 'rio de janeiro', 100% RJ); note '04482' has 0 rows
VERIFIED_ZIP_CITY_OVERRIDES = {
    ("87025", "PR"): "maringa",
    ("22790", "RJ"): "rio de janeiro",
}

print(f"Configured BRONZE_PATH           : {BRONZE_PATH}")
print(f"Configured SILVER_SCHEMA         : {SILVER_SCHEMA}")
print(f"Configured SILVER_TARGET         : {SILVER_TARGET_TABLE}")
print(f"Configured QUARANTINE_TARGET     : {QUARANTINE_TARGET_TABLE}")
print(f"Configured ZIP/City overrides    : {len(VERIFIED_ZIP_CITY_OVERRIDES)} pair(s)")
print(f"WRITE_ENABLED                    : {WRITE_ENABLED}")

## Section 3 — Read Bronze Parquet & Capture Source File Lineage
### شرح الدوال المستخدمة:
- `spark.read.format("parquet").load(BRONZE_PATH)`: تقرأ ملف Parquet الخام من طبقة Bronze مع الاحتفاظ بأنواع البيانات الفيزيائية.
- `F.input_file_name()`: دالة Spark مدمجة تضيف تلقائياً اسم ومسار الملف الفعلي الذي جاء منه كل صف؛ مفيدة جداً للتتبع (`Lineage`) ومعرفة مصدر أي خطأ لاحقاً.
- `df_bronze.cache()`: نثبّت الـ DataFrame في الذاكرة مؤقتاً (حجمه صغير جداً: 3,095 صفاً ~ 127 KB) لتجنب إعادة قراءة الملف من ADLS عدة مرات أثناء الفحص والعد.

In [ ]:
df_bronze = (
    spark.read
    .format("parquet")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

df_bronze.cache()
bronze_count = df_bronze.count()

print(f"Bronze Row Count: {bronze_count}")
df_bronze.printSchema()
# Uncomment during interactive review:
# display(df_bronze.limit(10))

## Section 4 — Stage 1: Schema Validation (Fail-Fast Contract Check)
### شرح الكود:
- قبل أي تنظيف، نتحقق أن الأعمدة الـ 4 الأساسية موجودة فعلياً في الملف (`seller_id`, `seller_zip_code_prefix`, `seller_city`, `seller_state`).
- إذا تغيّر اسم عمود أو اختفى في المصدر (`Schema Drift`)، يتوقف الدفتر فوراً برسالة خطأ واضحة (`raise ValueError`) بدلاً من إكمال معالجة ناقصة.

In [ ]:
REQUIRED_COLUMNS = [
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
]

missing_columns = [c for c in REQUIRED_COLUMNS if c not in df_bronze.columns]
if missing_columns:
    raise ValueError(f"Schema validation failed — missing required columns: {missing_columns}")

print("Stage 1 Schema Validation: PASSED")

## Section 5 — Stage 2: Text & Type Standardization (Without Hiding Data Errors)
### شرح كل دالة ولماذا اخترناها:
1. **`F.lower(F.trim(F.col("seller_id").cast("string")))`**: توحيد `seller_id` لحروف صغيرة وإزالة المسافات الطرفية لضمان تطابق الربط (`JOIN`) مع `order_items.seller_id`.
2. **`F.trim(F.col("seller_zip_code_prefix").cast("string"))` بدون `lpad` أعمى**:
   - **لماذا أزلنا `lpad(..., 5, '0')` قبل التحقق؟** لأن كل الرموز البريدية في المصدر تأتي بالفعل كـ 5 خانات (`varchar(5)`). استخدام `lpad` قبل التحقق قد يحوّل قيمة فاسدة مثل `'123'` إلى `'00123'` فتنجح في اختبار الـ 5 أرقام بالخطأ! نتحقق أولاً أنها 5 أرقام صالحة (`^\d{5}$`) ونحتفظ بها `string` لحماية الأصفار البادئة.
3. **`F.regexp_replace(..., r"\s+", " ")`**: دالة `F.trim()` تزيل المسافات من البداية والنهاية فقط، لكنها لا تزيل المسافات المزدوجة بالداخل (مثل `'sao  paulo'` و `'ferraz de  vasconcelos'` و `'sao  jose dos pinhais'`). لذا نستخدم `regexp_replace` لتحويل أي مسافات متكررة إلى مسافة واحدة.
4. **`seller_city_raw`**: نحتفظ بالنسخة الأصلية المنظفة من المسافات قبل أي تعديل أو تعويض حتى لا نفقد أي معلومة مصدرية.
5. **`F.upper(F.trim(F.col("seller_state")))`**: توحيد كود الولاية إلى حرفين كبيرين (`SP`, `RJ`, `MG`, ...).

In [ ]:
df_std = (
    df_bronze
    # Normalize seller_id to trimmed lowercase string
    .withColumn("seller_id", F.lower(F.trim(F.col("seller_id").cast("string"))))
    # Keep ZIP as trimmed string (NO blind lpad before validation!)
    .withColumn("seller_zip_code_prefix", F.trim(F.col("seller_zip_code_prefix").cast("string")))
    # Collapse internal multi-spaces, trim edges, and lowercase city; preserve raw copy
    .withColumn(
        "seller_city_raw",
        F.lower(F.trim(F.regexp_replace(F.col("seller_city").cast("string"), r"\s+", " ")))
    )
    # Standardize 2-letter Brazilian UF state code to uppercase
    .withColumn("seller_state", F.upper(F.trim(F.col("seller_state").cast("string"))))
)

print("Stage 2 Standardization complete. Distinct normalized cities:", df_std.select("seller_city_raw").distinct().count())

## Section 6 — Stage 3: Primary Key Uniqueness Check (BEFORE Any Filtering — NO `dropDuplicates`!)
### لماذا ننفذ هذا الفحص هنا وبدون `dropDuplicates()`؟
- **الخطأ الشائع في النوتبوك القديمة (`Seller_Transformation.ipynb`):** استدعاء `df.dropDuplicates()` قبل فحص تكرار `seller_id` يُخفي مشكلة التكرار في المصدر بدلاً من اكتشافها!
- هنا نستخدم `groupBy("seller_id").count().filter(F.col("count") > 1)` على البيانات قبل أي تصفية. إذا وُجد أي تكرار، يتوقف الدفتر فوراً للتحقيق بدلاً من اختيار صف عشوائي.

In [ ]:
duplicate_seller_ids = (
    df_std
    .filter(F.col("seller_id").isNotNull() & (F.col("seller_id") != ""))
    .groupBy("seller_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_pk_count = duplicate_seller_ids.count()
if duplicate_pk_count > 0:
    # display(duplicate_seller_ids.limit(20))
    raise ValueError(
        f"Primary Key Violation: Found {duplicate_pk_count} duplicate seller_id values. "
        "Stopping without blind dropDuplicates()."
    )

print("Stage 3 Primary Key Uniqueness Check: PASSED (0 duplicate seller_id)")

## Section 7 — Stage 4: Tier-1 Row-Level Entity Validation & Quarantine Split
### شرح الدوال والمنطق الهندسي:
- **متى يُعزل الصف بالكامل في `Quarantine`؟** فقط إذا فشل أحد شروط **الكيان (Entity)** الثلاثة:
  1. `seller_id` فارغ أو لا يطابق صيغة 32 خانة Hex (`rlike("^[0-9a-f]{32}$")`).
  2. `seller_zip_code_prefix` فارغ أو لا يطابق 5 أرقام بالضبط (`rlike("^\d{5}$")`).
  3. `seller_state` فارغ أو غير موجود ضمن قائمة الـ 27 ولاية برازيلية الرسمية (`isin(*VALID_BR_STATES)`).
- نستخدم `F.concat_ws(";", ...)` لدمج **كل أسباب الرفض معاً** في عمود `dq_reject_reason` (في حال اجتمع أكثر من خطأ في نفس الصف)، لأن `F.concat_ws` يتجاهل القيم `NULL` تلقائياً!
- على بيانات Olist الحالية: نتوقع `0` صفوف في `df_quarantine` و `3,095` صفاً في `df_valid_entities`.

In [ ]:
# Define strict boolean failure predicates for Tier-1 Entity fields
seller_id_invalid = (
    F.col("seller_id").isNull() |
    (F.col("seller_id") == "") |
    (~F.col("seller_id").rlike(r"^[0-9a-f]{32}$"))
)

zip_invalid = (
    F.col("seller_zip_code_prefix").isNull() |
    (~F.col("seller_zip_code_prefix").rlike(r"^\d{5}$"))
)

state_invalid = (
    F.col("seller_state").isNull() |
    (~F.col("seller_state").isin(*VALID_BR_STATES))
)

# Build auditable semicolon-separated rejection reason per row
df_classified = df_std.withColumn(
    "dq_reject_reason",
    F.concat_ws(
        ";",
        F.when(seller_id_invalid, F.lit("INVALID_SELLER_ID")),
        F.when(zip_invalid, F.lit("INVALID_ZIP_PREFIX")),
        F.when(state_invalid, F.lit("INVALID_BR_STATE")),
    )
)

# Split cleanly into Entity-Valid vs Row-Level Quarantine
df_quarantine = (
    df_classified
    .filter(F.col("dq_reject_reason") != "")
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumnRenamed("_source_file", "source_file")
    .withColumn("quarantined_at", F.current_timestamp())
)

df_valid_entities = df_classified.filter(F.col("dq_reject_reason") == "").drop("dq_reject_reason")

print("Tier-1 Entity Valid Count :", df_valid_entities.count())
print("Tier-1 Row Quarantine     :", df_quarantine.count())

## Section 8 — Stage 5: Tier-2 Attribute-Level City DQ (Option 2: Auditable Static Override + Safe NULL Fallback)
### ماذا تفعل هذه الخلية خطوة بخطوة؟
- البائعان ذوا القيمتين الفاسدتين (`vendas@creditparts.com.br` و `04482255`) يملكان **39 بند مبيعات حقيقي** في `order_items` (`7,532.41` BRL)، لذا يبقيان داخل `silver.sellers`.
- بدلاً من عمل `JOIN` مع `dim_geolocation` في هذه المرحلة (الخيار الأول)، ننفّذ **الخيار الثاني (Static Override مُوثّق للتعلّم والـ POC)**:
  1. **`CORRECTED_FROM_ZIP_LOOKUP` (متوقع: 2 صف):** إذا كانت قيمة المدينة فاسدة (إيميل أو أرقام فقط) ولها تطابق موثّق في `VERIFIED_ZIP_CITY_OVERRIDES` (بشَرط تطابق الـ ZIP والولاية معاً: `87025 + PR -> maringa` و `22790 + RJ -> rio de janeiro`)، نضع المدينة الصحيحة في `seller_city`، ونحفظ القيمة الأصلية في `seller_city_raw`، ونُسجّل المصدر في `seller_city_source = 'STATIC_ZIP_OVERRIDE_POC'`.
  2. **`INVALID_CITY_NULLIFIED` (متوقع: 0 حالياً — صمام أمان للمستقبل):** إذا وصلت مستقبلاً قيمة مدينة فاسدة غير موجودة في القاموس الموثّق، تتحوّل بأمان إلى `NULL` في `seller_city` مع `seller_city_source = 'NULLIFIED_INVALID_SOURCE'` حتى لا تتسرّب للتقارير.
  3. **`REVIEW_REQUIRED` (متوقع: 36 صفاً):** رصد كل المدن التي تحتوي على علامات مركبة (`/` `\` `,` `-` `(` `)`), أو أحرف غير ASCII/تشكيل (`são`, `d´oeste`), أو كود ولاية منفرد/ملحق (`sp`, `brasilia df`, `sao paulo sp`, `angra dos reis rj`, `aguas claras df`), أو الاختصار (`sbc`) — للمراجعة لاحقاً عبر City Mapping.
  4. **`OK` (متوقع: 3,057 صفاً):** مدن سليمة تماماً (بما فيها المدن البرازيلية الحقيقية من 3 أحرف مثل `itu`, `uba`, `poa`, `ipe`, `jau`).

In [ ]:
# Regex patterns for Tier-2 Attribute-level City DQ
EMAIL_REGEX = r"(?i)^[^@\s]+@[^@\s]+\.[^@\s]+$"
DIGITS_ONLY_REGEX = r"^\d+$"

# Expanded Review Regex (catches all 26 peer-review cases PLUS the 10 missed cases = 36 total)
UF_PATTERN = "ac|al|ap|am|ba|ce|df|es|go|ma|mt|ms|mg|pa|pb|pr|pe|pi|rj|rn|rs|ro|rr|sc|sp|se|to"
CITY_REVIEW_REGEX = rf"[/\\,\-\(\)]|[^\x00-\x7F]|^(?:{UF_PATTERN})$|\s(?:{UF_PATTERN})$|^sbc$"

city_corrupted = (
    F.col("seller_city_raw").isNull() |
    (F.col("seller_city_raw") == "") |
    F.col("seller_city_raw").rlike(EMAIL_REGEX) |
    F.col("seller_city_raw").rlike(DIGITS_ONLY_REGEX)
)

city_needs_review = (~city_corrupted) & F.col("seller_city_raw").rlike(CITY_REVIEW_REGEX)

# Build auditable static override expression from VERIFIED_ZIP_CITY_OVERRIDES (no cross-table join in Stage-1)
override_city_expr = F.lit(None).cast("string")
for (zip_code, state_code), canonical_city in VERIFIED_ZIP_CITY_OVERRIDES.items():
    override_city_expr = F.when(
        city_corrupted &
        (F.col("seller_zip_code_prefix") == F.lit(zip_code)) &
        (F.col("seller_state") == F.lit(state_code)),
        F.lit(canonical_city)
    ).otherwise(override_city_expr)

city_corrected = city_corrupted & override_city_expr.isNotNull()

df_city_cleaned = (
    df_valid_entities
    # Apply verified (ZIP, State) override for known corrupted cities; fallback to NULL for unknown future corrupted values
    .withColumn(
        "seller_city",
        F.when(city_corrected, override_city_expr)
         .when(city_corrupted, F.lit(None).cast("string"))
         .otherwise(F.col("seller_city_raw"))
    )
    # Classify every row transparently for downstream analytics & future City Reference Mapping
    .withColumn(
        "seller_city_dq_status",
        F.when(city_corrected, F.lit("CORRECTED_FROM_ZIP_LOOKUP"))
         .when(city_corrupted, F.lit("INVALID_CITY_NULLIFIED"))
         .when(city_needs_review, F.lit("REVIEW_REQUIRED"))
         .otherwise(F.lit("OK"))
    )
    # Explicit provenance tracking of how seller_city was populated
    .withColumn(
        "seller_city_source",
        F.when(city_corrected, F.lit("STATIC_ZIP_OVERRIDE_POC"))
         .when(city_corrupted, F.lit("NULLIFIED_INVALID_SOURCE"))
         .otherwise(F.lit("SOURCE_SELLERS_CLEAN"))
    )
)

df_city_cleaned.groupBy("seller_city_dq_status", "seller_city_source").count().show(truncate=False)

print("Preview of corrected seller_city rows (Auditable Static ZIP Override):")
(
    df_city_cleaned
    .filter(F.col("seller_city_dq_status") == "CORRECTED_FROM_ZIP_LOOKUP")
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_state",
        "seller_city_raw",
        "seller_city",
        "seller_city_dq_status",
        "seller_city_source",
    )
    .show(truncate=False)
)

## Section 9 — Stage 6: Build Final Silver DataFrame with Full Audit & Lineage
### ترتيب الأعمدة النهائي في عقد `silver.sellers`:
- **Business Columns:** `seller_id`, `seller_zip_code_prefix`, `seller_city`, `seller_state`.
- **Data Quality & Provenance:** `seller_city_raw`, `seller_city_dq_status`, `seller_city_source`.
- **Audit & Lineage:** `source_system`, `source_table`, `source_file`, `ingestion_run_id`, `silver_processed_at`.

In [ ]:
df_silver = (
    df_city_cleaned
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumnRenamed("_source_file", "source_file")
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "seller_city_raw",
        "seller_city_dq_status",
        "seller_city_source",
        "source_system",
        "source_table",
        "source_file",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

df_silver.printSchema()

## Section 10 — Stage 7: Pre-Write Reconciliation (Zero Silent Row Loss)
### شرط القبول الإلزامي:
يجب أن يتساوى عدد صفوف المصدر تماماً مع مجموع صفوف Silver + صفوف Quarantine (`bronze_count == silver_count + quarantine_count`)، مع التحقق من توزيع حالات جودة المدينة الأربعة (`OK + REVIEW_REQUIRED + CORRECTED_FROM_ZIP_LOOKUP + INVALID_CITY_NULLIFIED == silver_count`).

In [ ]:
silver_count = df_silver.count()
quarantine_count = df_quarantine.count()
corrected_city_count = df_silver.filter(F.col("seller_city_dq_status") == "CORRECTED_FROM_ZIP_LOOKUP").count()
nullified_city_count = df_silver.filter(F.col("seller_city_dq_status") == "INVALID_CITY_NULLIFIED").count()
review_city_count = df_silver.filter(F.col("seller_city_dq_status") == "REVIEW_REQUIRED").count()
ok_city_count = df_silver.filter(F.col("seller_city_dq_status") == "OK").count()

print(f"Bronze Input Count                  : {bronze_count}")
print(f"Silver Output Count                 : {silver_count} (Expected: 3095)")
print(f"Row-Level Quarantine Count          : {quarantine_count} (Expected: 0)")
print(f"  -> City OK                        : {ok_city_count} (Expected: 3057)")
print(f"  -> City REVIEW_REQUIRED           : {review_city_count} (Expected: 36)")
print(f"  -> City CORRECTED_FROM_ZIP_LOOKUP : {corrected_city_count} (Expected: 2)")
print(f"  -> City INVALID_CITY_NULLIFIED    : {nullified_city_count} (Expected: 0)")

if bronze_count != silver_count + quarantine_count:
    raise ValueError(
        f"Reconciliation FAILED: bronze({bronze_count}) != silver({silver_count}) + quarantine({quarantine_count})"
    )

dq_sum = ok_city_count + review_city_count + corrected_city_count + nullified_city_count
if dq_sum != silver_count:
    raise ValueError(
        f"City DQ partition FAILED: sum({dq_sum}) != silver({silver_count})"
    )

print("Stage 7 Reconciliation: PASSED")

## Section 11 — Proposed Silver & Quarantine Write (DISABLED BY DEFAULT)
> **DO NOT EXECUTE THE WRITE STEP UNTIL THE TRANSFORMATION HAS BEEN REVIEWED AND APPROVED.**

### لماذا نستخدم `CREATE SCHEMA IF NOT EXISTS silver` ثم `saveAsTable(SILVER_TARGET_TABLE)`؟
- في Lakehouse مُفعّل به Schemas (`bronze`/`silver`/`gold`)، يضمن `spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")` عدم ظهور خطأ `SCHEMA_NOT_FOUND` إذا لم يُنشأ Schema باسم `silver` مسبقاً في الـ Catalog.
- **ملاحظة مهمّة:** إذا ظهر لك خطأ متعلق بعدم تفعيل Schemas في هذا الـ Lakehouse، فهذا يعني أن الـ Lakehouse مُنشأ بوضع Non-Schema (بدون مجلدات Schemas تحت Tables)؛ عندها غيّر اسم الجدول فقط إلى `silver_sellers` أو استخدم Lakehouse مُفعّل به Schemas (`Retail_Analytics_Lakehouse`).
- تقوم `.saveAsTable("silver.sellers")` بكتابة ملفات Delta **وتسجيل الجدول تلقائياً في الـ Catalog** في خطوة ذرية واحدة.

In [ ]:
if WRITE_ENABLED:
    # Ensure target schema exists in schemas-enabled Fabric Lakehouse Catalog
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TARGET_TABLE)
    )
    print(f"Successfully wrote {silver_count} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        (
            df_quarantine.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(QUARANTINE_TARGET_TABLE)
        )
        print(f"Appended {quarantine_count} rows to {QUARANTINE_TARGET_TABLE}")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 12 — Post-Write Catalog Validation (Future Execution Procedure Only)
- بعد اعتماد التنفيذ الفعلي (`WRITE_ENABLED = True`)، نختبر الجدول بالقراءة من الـ **Catalog باسم الجدول (`spark.table(SILVER_TARGET_TABLE)`)** وليس فقط من المسار، للتحقق من:
  1. تطابق عدد الصفوف (`3,095`).
  2. صفر تكرار في `seller_id`.
  3. حفظ مدينتي `maringa` و `rio de janeiro` بشكل سليم في الصفّين المصحَّحين.
  4. لاحقاً (بعد بناء `silver.order_items`): التحقق عبر `LEFT ANTI JOIN` أن `0` صفوف في `silver.order_items` يتيمة من `silver.sellers`.
  5. تحرير ذاكرة الـ Cache بـ `df_bronze.unpersist()`.

In [ ]:
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    final_count = df_check.count()
    if final_count != silver_count:
        raise ValueError(f"Post-write count mismatch: expected {silver_count}, actual {final_count}")

    final_dups = df_check.groupBy("seller_id").count().filter(F.col("count") > 1).count()
    if final_dups > 0:
        raise ValueError(f"Post-write PK violation: {final_dups} duplicate seller_id found")

    print(f"POST-WRITE CATALOG VALIDATION PASSED: {final_count} rows, 0 duplicate seller_id.")
    print("Verifying Option 2 ZIP-corrected rows persisted in Catalog table:")
    (
        df_check
        .filter(F.col("seller_city_dq_status") == "CORRECTED_FROM_ZIP_LOOKUP")
        .select("seller_id", "seller_zip_code_prefix", "seller_state", "seller_city_raw", "seller_city", "seller_city_source")
        .show(truncate=False)
    )
    df_check.printSchema()
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")

df_bronze.unpersist()